In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from gensim.models import KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import re

In [2]:
df = pd.read_csv("train.tsv", sep="\t", header=None, names=["label", "text"], on_bad_lines='skip')

texts = df["text"].astype(str).tolist()
labels = df["label"].astype(int).tolist()

In [4]:
# https://github.com/sdadas/polish-nlp-resources?tab=readme-ov-file#word2vec
word2vec = KeyedVectors.load("word2vec/word2vec_100_3_polish.bin")

In [9]:
vec_size = 100

In [32]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-ząćęłńóśźż ]", " ", text)
    return text.split()

def text_to_vector(text):
    words = preprocess(text)
    vectors = [word2vec[word] for word in words if word in word2vec]
    if len(vectors) == 0:
        return np.zeros(vec_size)
    return np.mean(vectors, axis=0)

In [33]:
X_train_texts, X_test_texts, y_train, y_test = train_test_split(texts, np.array(labels), test_size=0.2, random_state=42)

X_train = np.array([text_to_vector(t) for t in X_train_texts])
X_test = np.array([text_to_vector(t) for t in X_test_texts])

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

In [34]:
model = nn.Sequential(
    nn.Linear(vec_size, 128),
    nn.ReLU(),
    nn.Linear(128, 256),
    nn.Linear(256, 64),
    nn.Linear(64, 1),
    nn.Sigmoid()
)

In [35]:
loss_f = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 10
batch_size = 64

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for i in range(0, len(X_train_tensor), batch_size):
        x_batch = X_train_tensor[i:i+batch_size]
        y_batch = y_train_tensor[i:i+batch_size]
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = loss_f(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 230.5145
Epoch 2, Loss: 179.1026
Epoch 3, Loss: 164.6435
Epoch 4, Loss: 155.2604
Epoch 5, Loss: 147.8014
Epoch 6, Loss: 142.0537
Epoch 7, Loss: 137.2816
Epoch 8, Loss: 131.9655
Epoch 9, Loss: 128.2232
Epoch 10, Loss: 124.3598


In [36]:
model.eval()
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
with torch.no_grad():
    outputs = model(X_test_tensor)
    predictions = outputs.numpy().round()
    acc = accuracy_score(y_test, predictions)
    print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9410


# Samples

In [40]:
X_test_texts[:5]

['Zagraniczne media po meczu PPD Zagrzeb - Vive: genialny Ivić w starciu ze "swoimi", Vive znów na zwycięskiej ścieżce W spotkaniu 7. kolejki Ligi Mistrzów, polskie Vive Tauron Kielce pokonało na wyjeździe chorwacki RK Prvo plinarsko drustvo Zagrzeb 26:23. Co o tym meczu piszą zagraniczne media?',
 'Oficjalnie: duże wzmocnienie Miasta Szkła Krosno Dino Pita został oficjalnie nowym koszykarzem Miasta Szkła Krosno. 28-letni reprezentant Szwecji związał się z klubem do końca sezonu.',
 'Fortuna I liga: zmarnowana szansa Sandecji. Remis GieKSy po zmianie trenera Remis 1:1 z Podbeskidziem Bielsko-Biała nie pozwolił Sandecji Nowy Sącz przesunąć się na miejsce premiowane awansem. Identycznym wynikiem zakończył się mecz w Katowicach.',
 'Nie żyje Morten Aagheim, były reprezentant Norwegii w skokach narciarskich W wieku niespełna 37 lat zmarł Morten Aagheim. Na przełomie wieków był jednym z czołowych norweskich skoczków narciarskich i wielokrotnie plasował się w pierwszej dziesiątce konkursów P

In [41]:
predictions[:5]

array([[1.],
       [1.],
       [1.],
       [0.],
       [0.]], dtype=float32)